# 1. RAG: Ask MongoDB 9.0 release notes

### Scenario
> *A customer just upgraded to MongoDB 9.0. They report: slow aggregations, memory errors on writes, and change-stream lag.*

### What this notebook shows
**RAG = Retrieve, then Generate.** We embed the document and the question into the same vector space, find the most similar sections, and ask a local LLM to answer from ONLY that context.

### The setup on screen
- This notebook on the left
- `9.0 Upcoming.pdf` open on the right (visual reference)

Run cells top to bottom.

## Setup — imports and settings

In [10]:
import ollama
# Ollama runs everything locally — LLM + embeddings, no cloud.

import numpy as np
# Numpy for vector math (cosine similarity).

EMBED_MODEL = "nomic-embed-text"
# Free local embedding model — 274MB, no API key.
# Embeddings turn text into vectors that capture meaning, not just keywords.
#
# You can swap this with MongoDB Voyage AI embeddings:
# Voyage models are purpose-built for RAG — better retrieval quality

CHAT_MODEL = "qwen2.5:3b"
# The LLM we ask questions to.

DOC_PATH = "9.0_notes.md"
# The MongoDB 9.0 release notes.

print(f"Embedding model: {EMBED_MODEL}")
print(f"Chat model: {CHAT_MODEL}")
print(f"Source: {DOC_PATH}")

Embedding model: nomic-embed-text
Chat model: qwen2.5:3b
Source: 9.0_notes.md


## Step 1 — Read the markdown file

In [11]:
with open(DOC_PATH, encoding="utf-8") as f:
    doc_text = f.read()
# open the file.

print(f"Characters: {len(doc_text)}")
print(f"\nFirst 250 chars:\n{doc_text[:250]}")

Characters: 7649

First 250 chars:
# Release Notes for MongoDB 9.0

**Important: MongoDB 9.0 Availability**

MongoDB 9.0 is production ready and is being incrementally rolled out to MongoDB Atlas clusters that use the Latest Version With Auto Upgrades option.

Availability for the rem


## Step 2 — Split into chunks and embed them

We slide a 1500-character window across the document, then turn each chunk into a vector using `nomic-embed-text`. These vectors capture semantic meaning — similar ideas get similar vectors.

In [12]:
chunk_size, overlap = 1500, 300
# 1500 chars per chunk, 300 overlap.

chunks = []
for i in range(0, len(doc_text), chunk_size - overlap):
    piece = doc_text[i:i + chunk_size].strip()
    if len(piece) > 80:
        chunks.append(piece)

print(f"Chunks: {len(chunks)}  |  sending to embedding model...")
# Few chunks because the document is short (~7K chars).

# Embed each chunk — this takes a few seconds
chunk_vecs = []
for i, c in enumerate(chunks):
    resp = ollama.embeddings(model=EMBED_MODEL, prompt=c)
    chunk_vecs.append(resp["embedding"])
    print(f"  chunk {i+1}/{len(chunks)} embedded")
# Each chunk is now a vector of 768 numbers — its meaning fingerprint.

print(f"\nEach vector has {len(chunk_vecs[0])} dimensions")
# 768 dimensions = nomic-embed-text's output size.

Chunks: 7  |  sending to embedding model...
  chunk 1/7 embedded
  chunk 2/7 embedded
  chunk 3/7 embedded
  chunk 4/7 embedded
  chunk 5/7 embedded
  chunk 6/7 embedded
  chunk 7/7 embedded

Each vector has 768 dimensions


## Step 3 — First, ask the model WITHOUT any context

We ask a question about MongoDB 9.0 *without* giving the model any documents. The model was trained at a cutoff date — it doesn't know about MongoDB 9.0 because 9.0 didn't exist yet. Watch what happens.

In [13]:
raw_question = "What is the per-operation memory limit introduced in MongoDB 9.0?"
# This question references 9.0 — a version the model hasn't seen.

raw_response = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content": "Answer ONLY if you know. If you are unsure, say so."},
        {"role": "user", "content": raw_question},
    ],
)
# No context provided — the model is flying blind.

print(f"QUESTION: {raw_question}\n")
print("MODEL (no context):")
print(raw_response["message"]["content"].strip())
# The model likely doesn't know — this proves it needs RAG.
# Training data cutoff means it has no knowledge of MongoDB 9.0.

print("\n---")
print("See? Without context, the model is guessing or admitting ignorance.")
print("Now we'll use RAG to give it the actual 9.0 release notes.")

QUESTION: What is the per-operation memory limit introduced in MongoDB 9.0?

MODEL (no context):
I'm unsure about the specific per-operation memory limit introduced in MongoDB 9.0. MongoDB has made improvements and changes to its memory management over the years, but I don't have information on the exact per-operation memory limit introduced in MongoDB 9.0. For the most accurate and up-to-date information, you should refer to the official MongoDB documentation or press releases from the company.

---
See? Without context, the model is guessing or admitting ignorance.
Now we'll use RAG to give it the actual 9.0 release notes.


## Step 4 — Embed the question + find the best chunks

We embed the question into the same vector space, then use cosine similarity to find which chunks are closest in meaning.

In [14]:
question = "What changes in MongoDB 9.0 could cause slow aggregations or memory errors?"
# The exact question a TS engineer would ask when diagnosing a 9.0 upgrade.

# Embed the question
q_resp = ollama.embeddings(model=EMBED_MODEL, prompt=question)
q_vec = q_resp["embedding"]
# The question is now in the same vector space as the chunks.

# Cosine similarity: measures the angle between two vectors. 1.0 = same direction.
def cos_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

scores = [cos_sim(q_vec, v) for v in chunk_vecs]
top_indices = np.argsort(scores)[::-1][:5]
# Sort by score descending, take top 5.

print(f"Question: {question}\n")
print("Top chunks by relevance:")
for i, idx in enumerate(top_indices):
    print(f"\n--- Chunk {i+1} (score: {scores[idx]:.3f}) ---")
    print(chunks[idx][:250])

# Combine top chunks into one context
context = "\n\n---\n\n".join(chunks[idx] for idx in top_indices)
print(f"\nTotal context: {len(context)} chars across 5 chunks")

Question: What changes in MongoDB 9.0 could cause slow aggregations or memory errors?

Top chunks by relevance:

--- Chunk 1 (score: 0.765) ---
# Release Notes for MongoDB 9.0

**Important: MongoDB 9.0 Availability**

MongoDB 9.0 is production ready and is being incrementally rolled out to MongoDB Atlas clusters that use the Latest Version With Auto Upgrades option.

Availability for the rem

--- Chunk 2 (score: 0.739) ---
a Storage

Starting in 9.0, MongoDB stores time series collections as a single namespace that contains time series data compressed into buckets. Time series collections are no longer writable non-materialized views.

### renameCollection

Starting in

--- Chunk 3 (score: 0.731) ---
rs open cursors only on the shards that hold relevant data and adjust targeting automatically as the cluster topology changes. MongoDB 9.0 also adds the `ignoreRemovedShards` parameter to the `$changeStream` aggregation stage. When set to `true`, a c

--- Chunk 4 (score: 0.712) ---
eration

## Step 5 — Show the exact context the model receives

This is the retrieved text we're about to send to the LLM. The model will answer from ONLY this.

In [15]:
print(context)
# This is what the model sees — nothing else. 

# Release Notes for MongoDB 9.0

**Important: MongoDB 9.0 Availability**

MongoDB 9.0 is production ready and is being incrementally rolled out to MongoDB Atlas clusters that use the Latest Version With Auto Upgrades option.

Availability for the remainder of MongoDB Atlas, MongoDB Enterprise Advanced, and MongoDB Community is coming soon.

## General Changes

### Change Stream Pre-Image Metrics

Starting in MongoDB 9.0, the `changeStreamPreImages.purgingJob.docsDeleted` and `changeStreamPreImages.purgingJob.bytesDeleted` serverStatus metrics may report estimates rather than exact values. These estimates occur when size and count information is not available for the `config.system.preimages` collection.

### Multi-Document Transaction Limit

Starting in MongoDB 9.0, the server limits the number of concurrently open multi-document transactions for external clients. When the number of open transactions reaches the limit set by the `maxConcurrentMultiDocumentTransactions` parameter (defau

## Step 6 — Ask the LLM (using ONLY the context above)

In [16]:
response = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content":
            "You are a MongoDB support engineer. "
            "Answer ONLY using the provided context below. "
            "If the answer is not in the context, say 'Not found in the release notes.' "
            "Be specific: mention feature names, error codes, and versions. "
            "Keep answers under 5 bullet points."},
        {"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION: {question}"},
    ],
)
# Sends the system prompt + context + question to the local LLM.

answer = response["message"]["content"].strip()

print("MODEL ANSWER:\n")
print(answer)
# The model answered ONLY from the retrieved context — that's RAG.

MODEL ANSWER:

- The 1 gigabyte or 20% of the memory available to the server process, whichever is greater, is now the per-operation memory limit that MongoDB 9.0 enforces. Operations exceeding this limit fail with error code 146 (ExceededMemoryLimit) or error code 292 (QueryExceededMemoryLimitNoDiskUseAllowed).
- MongoDB 9.0 introduces a per-operation memory limit of 1 gigabyte or 20% of the available memory to the server process, whichever is greater. Operations exceeding this limit fail with error code 146 (ExceededMemoryLimit) or error code 292 (QueryExceededMemoryLimitNoDiskUseAllowed).


## Step 7 — Ask another question (optional)

Try your own question — anything about MongoDB 9.0.

In [17]:
question2 = "What changed about change streams in MongoDB 9.0?"
# Another common support question.

q2_resp = ollama.embeddings(model=EMBED_MODEL, prompt=question2)
q2_vec = q2_resp["embedding"]
scores2 = [cos_sim(q2_vec, v) for v in chunk_vecs]
top2 = np.argsort(scores2)[::-1][:5]
context2 = "\n\n---\n\n".join(chunks[i] for i in top2)
# Same process: embed question, find closest chunk vectors, retrieve top 5.

response2 = ollama.chat(
    model=CHAT_MODEL,
    messages=[
        {"role": "system", "content":
            "You are a MongoDB support engineer. Answer ONLY using the provided context."},
        {"role": "user", "content": f"CONTEXT:\n{context2}\n\nQUESTION: {question2}"},
    ],
)

print(f"QUESTION: {question2}\n")
print("ANSWER:")
print(response2["message"]["content"].strip())
# The model answers from only the retrieved chunks — no hallucinating.

QUESTION: What changed about change streams in MongoDB 9.0?

ANSWER:
In MongoDB 9.0, change streams on sharded clusters open cursors only on the shards that hold relevant data and adjust targeting automatically as the cluster topology changes. Additionally, MongoDB 9.0 introduced the `ignoreRemovedShards` parameter to the `$changeStream` aggregation stage. When set to `true`, a change stream continues and skips events from a shard that was removed from the cluster instead of returning an error.


## Recap

- We **embedded** the document and question into the same vector space (nomic-embed-text)
- We **retrieved** the 5 most semantically similar chunks (cosine similarity)
- We **generated** an answer from ONLY those chunks
- That's RAG: **Retrieve → Augment → Generate**
- Embeddings beat TF-IDF because they understand meaning, not just keywords

**Next:** Notebook 2 gives the model tools to investigate on its own.